# Chapter 3 · 판매량 RF·XGBoost

**연구질문:** 판매량 예측에서 RF·부스팅은 프로모션·요일 정보를 활용해 기준선을 이기는가?

> 위에서 아래로 실행하면 끝까지 동작합니다. 온라인 실패 시 합성 대체(`DATA_KIND` 표시).

## 1. 핵심 개념·수식

**지도학습 형태의 예측:** $\hat y_{t}=f(y_{t-1},\dots,y_{t-L},\ \text{rolling}_{t-1},\ \text{calendar}_t,\ x_t)$. 모든 특성은 **예측 시점 $t$에 알 수 있는 정보**만 사용한다(롤링 통계는 반드시 한 칸 이동).

**시간순 검증:** 무작위 분할 대신 훈련 구간이 항상 검증 구간보다 앞서는 expanding-window 분할을 쓴다.

**기준선 대비 성능:** $\mathrm{MASE}<1$이면 1단계 나이브보다 낫다. 수익률처럼 예측력이 약한 자료는 **표본 외 $R^2_{OS}=1-\dfrac{\sum(y_t-\hat y_t)^2}{\sum(y_t-\bar y_{t-1})^2}$** ($\bar y_{t-1}$: 직전까지의 평균)로 평가한다.

**변수 중요도:** 순열 중요도는 특성을 섞었을 때 검증 오차가 늘어난 정도이며 인과적 중요도가 아니다. 상관된 특성이 있으면 해석이 불안정하다.

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
def aggregate_online_retail(df):
    # UCI Online Retail II: 취소 송장(번호가 C로 시작)과 음수 수량을 제외한 일별 매출
    df = df.copy(); df.columns = [str(c).strip() for c in df.columns]
    inv = df["Invoice"].astype(str)
    df = df[~inv.str.startswith("C") & (df["Quantity"] > 0) & (df["Price"] > 0)]
    rev = (df["Quantity"] * df["Price"]).groupby(pd.to_datetime(df["InvoiceDate"]).dt.normalize()).sum()
    s = rev.asfreq("D", fill_value=0.0)
    if len(s) < 200:
        raise ValueError("관측 부족")
    return s

def synthetic_sales(n=730, seed=SEED):
    rng = np.random.default_rng(seed); t = np.arange(n); dow = (t % 7)
    promo = np.zeros(n); 
    for s0 in range(40, n - 7, 60): promo[s0:s0 + 5] = 1.0
    mu = 1000 + 0.6 * t + 150 * np.sin(2 * np.pi * t / 365.25) - 200 * (dow >= 5) + 350 * promo
    return pd.DataFrame({"sales": np.maximum(mu + rng.normal(0, 60, n), 0), "promo": promo},
                        index=pd.date_range("2023-01-01", periods=n, freq="D"))

def load_business(force_synth=False):
    s, kind = None, "SYNTHETIC (NOT real data): 요일·연간 계절성과 프로모션이 있는 일별 매출"
    if USE_LIVE_DATA and not force_synth and ensure("openpyxl"):
        try:
            import zipfile
            z = zipfile.ZipFile(io.BytesIO(fetch_bytes("https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip", timeout=180)))
            name = [n for n in z.namelist() if n.lower().endswith(".xlsx")][0]
            sheets = pd.read_excel(io.BytesIO(z.read(name)), sheet_name=None)
            s = aggregate_online_retail(pd.concat(sheets.values(), ignore_index=True))
            kind = "REAL: UCI Online Retail II 일별 매출 (CC BY 4.0, 취소 송장 제외)"
        except Exception as e:
            print("Online Retail II 수집 실패 -> 합성 대체:", type(e).__name__, str(e)[:120])
    if s is not None:
        return dict(y=s.rename("sales"), m=7, freq="D", label="일별 매출", unit="GBP", kind=kind)
    df = synthetic_sales()
    return dict(y=df["sales"], df=df, m=7, freq="D", label="일별 매출(합성)", unit="원", kind=kind)

D = load_business(False)
y, DATA_KIND, M, FREQ = D["y"], D["kind"], D["m"], D["freq"]
print("DATA_KIND:", DATA_KIND); print("관측 수:", len(y), "| 기간:", y.index.min().date(), "~", y.index.max().date())

## 2. 특성 구성·모형·평가

In [ ]:
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import (RandomForestRegressor, HistGradientBoostingRegressor,
                              RandomForestClassifier, HistGradientBoostingClassifier)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

def calendar_features(idx, freq):
    out = pd.DataFrame(index=idx)
    if freq in ("D", "B", "h"):
        out["dow"] = idx.dayofweek
    if freq == "h":
        out["hour"] = idx.hour
    out["month_sin"] = np.sin(2 * np.pi * idx.month / 12); out["month_cos"] = np.cos(2 * np.pi * idx.month / 12)
    return out

def lag_features(s, lags, rolls=()):
    X = pd.DataFrame({f"lag_{k}": s.shift(k) for k in lags})
    for w in rolls:
        X[f"rmean_{w}"] = s.shift(1).rolling(w).mean()          # shift(1): 현재 값을 쓰지 않는다(누수 방지)
        X[f"rstd_{w}"] = s.shift(1).rolling(w).std()
    return X

def get_models():
    models = {
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
        "RandomForest": RandomForestRegressor(n_estimators=150, min_samples_leaf=5, max_features=0.6, n_jobs=-1, random_state=SEED),
        "HistGB": HistGradientBoostingRegressor(max_iter=250, learning_rate=0.05, random_state=SEED),
    }
    if ensure("xgboost"):                                         # 선택: 설치돼 있으면 XGBoost도 비교
        from xgboost import XGBRegressor
        models["XGBoost"] = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                                         random_state=SEED, n_jobs=-1, verbosity=0)
    return models

def fit_eval(X, target, n_test, models=None):
    # 앞쪽으로 학습, 마지막 n_test 관측으로 평가 (1단계 예측, 시차 입력은 실제값)
    data = pd.concat([X, target.rename("target")], axis=1).dropna()
    tr, te = data.iloc[:-n_test], data.iloc[-n_test:]
    Xtr, ytr, Xte, yte = tr.drop(columns="target"), tr["target"], te.drop(columns="target"), te["target"]
    models = models or get_models(); preds = {}
    for name, mdl in models.items():
        mdl.fit(Xtr, ytr); preds[name] = pd.Series(mdl.predict(Xte), index=te.index)
    return dict(Xtr=Xtr, ytr=ytr, Xte=Xte, yte=yte, preds=preds, models=models, data=data)

def cv_rmse(X, target, models, n_splits=4, test_size=None):
    data = pd.concat([X, target.rename("target")], axis=1).dropna()
    test_size = test_size or max(30, len(data) // 10)
    tss = TimeSeriesSplit(n_splits=n_splits, test_size=test_size); out = {k: [] for k in models}
    from sklearn.base import clone
    for tr_i, te_i in tss.split(data):
        a, b = data.iloc[tr_i], data.iloc[te_i]
        for name, mdl in models.items():
            m_ = clone(mdl).fit(a.drop(columns="target"), a["target"])
            out[name].append(rmse(b["target"], m_.predict(b.drop(columns="target"))))
    return pd.DataFrame(out).agg(["mean", "std"]).T.rename(columns={"mean": "CV_RMSE_mean", "std": "CV_RMSE_sd"})

In [ ]:
# 결과표: 기준선과 모델, 마지막 구간 평가 + 시간순 CV
def report(res, baselines, train_for_scale):
    rows = {}
    for name, f in {**baselines, **res["preds"]}.items():
        rows[name] = {"MAE": mae(res["yte"], f), "RMSE": rmse(res["yte"], f),
                      "MASE": mase(res["yte"], f, train_for_scale, m=1)}
    tab = pd.DataFrame(rows).T.sort_values("RMSE"); show(tab); return tab

In [ ]:
df_s = D.get("df")
s = y.astype(float)
parts = [lag_features(s, [1, 7, 14, 28], rolls=(7, 28)), calendar_features(s.index, "D")]
if df_s is not None and "promo" in df_s:
    parts.append(df_s[["promo"]])                   # 프로모션은 사전 계획되어 알려져 있다고 가정(합성자료)
else:
    print("프로모션 변수가 없는 실제 자료입니다 -> 요일·계절·시차만 사용합니다.")
X = pd.concat(parts, axis=1)
n_test = 120
res = fit_eval(X, s, n_test)
yte = res["yte"]
base = {"naive(lag1)": s.shift(1).loc[yte.index], "seasonal naive(lag7)": s.shift(7).loc[yte.index]}
tab = report(res, base, res["ytr"].to_numpy())
print("\n시간순 CV (RMSE):"); show(cv_rmse(X, s, get_models(), n_splits=4, test_size=60))
best = tab.drop(index=list(base), errors="ignore").index[0]
pi = permutation_importance(res["models"][best], res["Xte"], yte, n_repeats=5, random_state=SEED,
                            scoring="neg_root_mean_squared_error")
show(pd.Series(pi.importances_mean, index=res["Xte"].columns).sort_values(ascending=False).head(8).to_frame("RMSE 증가량"))

**읽는 법:** 프로모션 변수는 사전에 알려져 있다고 가정했다. 실제로 프로모션 일정이 사후에 정해지거나 판매에 반응해 바뀌면 이 가정이 깨진다(Ch.7 누수).

## 3. 결과 해석 작성
1. 기준선(나이브·계절 나이브·역사평균)보다 나은 모델이 있는가? 개선폭은 CV 변동성($sd$) 대비 큰가?
2. 변수 중요도가 도메인 지식과 일관되는가? (인과적 해석 금지)
3. 가장 위험한 누수 경로 3개를 적고, 이 노트북에서 어떻게 막았는지(롤링 shift, 학습구간 통계 스케일, 시간순 분할)를 서술한다.
4. `DATA_KIND`가 합성이면 결과는 **방법 점검**이다. 실제 자료로 바꿨을 때 달라질 점을 적는다.

**제출:** 결과표(기준선 포함), CV 표, 중요도 그래프/표, 해석 단락, 한계.